# 12. 지수 검증 (⑥)

외부 정답이 없는 대회 → **내적 타당도**로 신뢰도 확보.
1. **수렴타당도**: HSI(표면+LST)와 **독립 방법 UTCI**(복사물리)가 순위 일치하나 (Spearman)
2. **민감도/강건성**: 가중치 ±흔들어도 취약 상위 격자가 유지되나 (Jaccard·Spearman)
3. **알려진집단**: 표면유형별 HSI 단조성 (도로>주거>공원>수역)

(외부 대조=무더위쉼터/그늘막 위치 vs HSI는 데이터 수집 후 별도)

In [1]:
# ── 입력 존재 가드 ─────────────────────────────────────────────
# 이 노트북은 07 UTCI 의 산출물을 입력으로 씁니다.
# 해당 단계가 환경 제약으로 건너뛰어졌고 동봉 산출물도 없으면 아래 셀을 건너뜁니다.
import pathlib
_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
LOCAL_RUN = (_ROOT / "data/results/heat_index_utci.geojson").exists()
print("[실행]" if LOCAL_RUN else "[건너뜀]", "data/results/heat_index_utci.geojson", "-", "입력 있음" if LOCAL_RUN else "입력 없음 (07 UTCI 산출물 필요)")


[실행] data/results/heat_index_utci.geojson - 입력 있음


In [2]:
if LOCAL_RUN:
    import sys, pathlib
    sys.path.append(str(pathlib.Path.cwd().parent))
    import numpy as np, pandas as pd, geopandas as gpd
    from scipy.stats import spearmanr
    from src.config import DATA_RESULTS, DATA_PROCESSED

    h = gpd.read_file(DATA_RESULTS / "heat_index_final.geojson")[["grid_id", "hsi", "zoneName", "blockType"]]
    u = gpd.read_file(DATA_RESULTS / "heat_index_utci.geojson")[["grid_id", "utci_daymean"]]
    gf = gpd.read_file(DATA_PROCESSED / "grid_features.geojson")[["grid_id", "lst_peak", "imperv_frac", "tree_frac", "water_frac", "ndvi"]]
    d = h.merge(u, on="grid_id").merge(gf, on="grid_id").dropna(subset=["hsi", "utci_daymean"])

    print("① 수렴타당도 (Spearman ρ):")
    print(f"  HSI vs UTCI(독립 복사물리): {spearmanr(d.hsi, d.utci_daymean).statistic:.3f}")
    print(f"  HSI vs LST_peak:           {spearmanr(d.hsi, d.lst_peak).statistic:.3f}")
    print(f"  HSI vs NDVI(식생, 음이어야): {spearmanr(d.hsi, d.ndvi).statistic:.3f}")

① 수렴타당도 (Spearman ρ):
  HSI vs UTCI(독립 복사물리): 0.715
  HSI vs LST_peak:           0.792
  HSI vs NDVI(식생, 음이어야): -0.564


In [3]:
if LOCAL_RUN:
    # ② 민감도: 가중치 변형 → 취약 상위 10% 유지율(Jaccard)·순위상관
    def norm(s):
        lo, hi = s.quantile(0.02), s.quantile(0.98)
        return ((s - lo) / (hi - lo)).clip(0, 1)

    def hsi(wh, wi, wc):
        raw = wh * norm(d.lst_peak) + wi * d.imperv_frac - wc * (0.6 * d.tree_frac + 0.4 * d.water_frac)
        return norm(raw)

    base = hsi(0.5, 0.5, 0.5)
    top0 = set(d.index[base >= base.quantile(0.9)])
    print("② 민감도 (가중치 변형 vs 기본):")
    # 주의: 가중치의 일괄 상수배(예 0.33/0.33/0.33 = 0.66×기준)는 norm()이 흡수해
    # 순위가 항등으로 보존되므로 민감도 증거가 아님 → 실질 변형 4조합만 사용.
    for wh, wi, wc in [(0.6, 0.4, 0.5), (0.4, 0.6, 0.5), (0.5, 0.5, 0.3), (0.5, 0.5, 0.7)]:
        v = hsi(wh, wi, wc)
        topv = set(d.index[v >= v.quantile(0.9)])
        jac = len(top0 & topv) / len(top0 | topv)
        rho = spearmanr(base, v).statistic
        print(f"  heat{wh} imperv{wi} cool{wc}: 순위ρ {rho:.3f}, 상위10% 유지 {jac:.2f}")

② 민감도 (가중치 변형 vs 기본):
  heat0.6 imperv0.4 cool0.5: 순위ρ 0.996, 상위10% 유지 0.85
  heat0.4 imperv0.6 cool0.5: 순위ρ 0.997, 상위10% 유지 0.85


  heat0.5 imperv0.5 cool0.3: 순위ρ 0.998, 상위10% 유지 0.99
  heat0.5 imperv0.5 cool0.7: 순위ρ 0.999, 상위10% 유지 0.99


In [4]:
if LOCAL_RUN:
    # ③ 알려진집단: 표면유형별 단조성(보조) + 준독립 집단(05 blockType) 교차 검증
    def dom(row):
        fr = {"불투수": row.imperv_frac, "산림": row.tree_frac, "수역": row.water_frac}
        return max(fr, key=fr.get) if max(fr.values()) > 0.4 else "혼합"
    d["surf"] = d.apply(dom, axis=1)
    m = d.groupby("surf")["hsi"].mean()
    print("③-1 지배표면별 HSI (주의: 집단 정의가 HSI 입력과 겹침 - 보조 지표):")
    print(m.round(3).sort_values(ascending=False))
    ok1 = (m["불투수"] > m["혼합"]) and (m["혼합"] > max(m["산림"], m["수역"]))
    print("  판정: 불투수 > 혼합 > 냉각면(산림·수역) ->", "성립" if ok1 else "불성립")

    # ③-2 준독립 집단: 계획 용도(blockType)는 HSI 계산에 미사용 -> 순환성 없는 대조
    bt = d.groupby("blockType")["hsi"].agg(["mean", "size"])
    bt = bt[bt["size"] >= 200]
    hot = bt.loc[[i for i in ["중심상업", "근린상업", "아파트"] if i in bt.index], "mean"]
    cool = bt.loc[[i for i in ["공원", "근린공원", "녹지"] if i in bt.index], "mean"]
    ok2 = hot.min() > cool.max()
    print("③-2 blockType 대조: 상업·주거 %.2f~%.2f > 공원·녹지 %.2f~%.2f ->" % (hot.min(), hot.max(), cool.min(), cool.max()), "성립" if ok2 else "불성립")
    print()
    print("종합: ρ(UTCI) 0.72 수렴 / 실질 4조합 민감도 0.996~0.999 / 집단 단조성", "충족" if (ok1 and ok2) else "부분 충족")

③-1 지배표면별 HSI (주의: 집단 정의가 HSI 입력과 겹침 - 보조 지표):
surf
불투수    0.770
혼합     0.382
수역     0.290
산림     0.098
Name: hsi, dtype: float64
  판정: 불투수 > 혼합 > 냉각면(산림·수역) -> 성립
③-2 blockType 대조: 상업·주거 0.74~0.89 > 공원·녹지 0.24~0.31 -> 성립

종합: ρ(UTCI) 0.72 수렴 / 실질 4조합 민감도 0.996~0.999 / 집단 단조성 충족
